# CSCI 5622 HW #4 - Questions (a) and (b)

In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from transformers import BertTokenizer, BertModel
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer
import os

## (a) Data processing

In [3]:
labels_path = r"../Study 1 (EDAIC)/DepressionLabels.csv"
transcripts_path = r"../Study 1 (EDAIC)/EDAIC Transcripts"

# Read depression labels
labels_df = pd.read_csv(labels_path)
labels_df

,Participant_ID,PHQ_Score
0,300,2
1,301,3
2,302,4
3,303,0
4,304,6
...,...,...
214,698,19
215,702,0
216,703,8
217,707,1


### Identify participants for which data is available

In [45]:
# Check that labels provided have a matching transcript
# Disregard participant data where a label/score or transcript are unavailable
ids = list()
transcripts_list = list()
phq_scores = list()
all_scores = labels_df['PHQ_Score'].to_numpy()
for i, id in enumerate(labels_df['Participant_ID'].to_numpy()):
    if os.path.isfile(f'{transcripts_path}/{str(id)}_Transcript.csv'):
        ids.append(id)
        transcripts_list.append(f'{str(id)}_Transcript.csv')
        phq_scores.append(all_scores[i])
print(len(ids))

134


### Gather language features for each transcript

In [5]:
def read_transcript_csv(path):
    with open(path, 'r') as f:
        lines = f.readlines()
    processed_lines = [l.strip() for l in lines[1:]]
    return processed_lines

def compute_transcript_sentiment(lines:list):
    '''
    Code written using Microsoft Copilot
    Calculates average of each sentiment score as given by the Vader sentiment package
    '''
    analyzer = SentimentIntensityAnalyzer()
    scores = [analyzer.polarity_scores(statement) for statement in lines]
    
    # Aggregate by mean
    avg_compound = sum(s['compound'] for s in scores) / len(scores)
    avg_pos = sum(s['pos'] for s in scores) / len(scores)
    avg_neg = sum(s['neg'] for s in scores) / len(scores)
    avg_neu = sum(s['neu'] for s in scores) / len(scores)
    
    return {
        'avg_compound': avg_compound,
        'avg_pos': avg_pos,
        'avg_neg': avg_neg,
        'avg_neu': avg_neu
    }

def get_transcript_embeddings(lines:list):
    '''
    Code written using Microsoft Copilot
    Get average embedding of a transcript using pretrained BERT by mean pooling all statement embeddings 
    '''
    # Define BERT model
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    model = BertModel.from_pretrained('bert-base-uncased')

    embeddings = list()
    for statement in lines: # Get embedding for each statement
        inputs = tokenizer(statement, return_tensors='pt', truncation=True, padding=True)
        with torch.no_grad():
            outputs = model(**inputs)
        # Get [CLS] token embedding (first token)
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # shape: [1, hidden_size]
        embeddings.append(cls_embedding)  

    # Calculate average embedding across all statements
    transcript_embedding = torch.mean(torch.cat(embeddings, dim=0), dim=0)
    return transcript_embedding

#### Extract sentiment scores and BERT embeddings

In [13]:
# Iterate through all relevant CSV files
lang_features = dict()
all_transcripts = list()
for file in transcripts_list:
    # Read transcript and prepare for simple vectorization
    ls = read_transcript_csv(os.path.join(transcripts_path, file))
    all_transcripts.append(ls)
    
    # Get average sentiment scores using Vader
    sentiment_scores = compute_transcript_sentiment(ls)
    
    # Extract average embedding for each transcript using pretrained BERT
    embedding = get_transcript_embeddings(ls).numpy()

    # Store language features based on ID
    id = int(file[:3])
    tmp_features = {"sentiment":sentiment_scores, "embedding":embedding}
    lang_features[id] = tmp_features

#### Get syntactic vector of all transcripts

In [59]:
# Consolidate transcripts as contiguous lists (documents)
transcripts_concat = [' '.join(t) for t in all_transcripts]

# Vectorize
vectorizer = TfidfVectorizer(min_df=10)  # Reduce sparsity by requiring words appear in at least 10 documents
tfidf_matrix = vectorizer.fit_transform(transcripts_concat)
pure_tfidf_matrix = tfidf_matrix.toarray()

# Combine results with previous language features
for idx, id_ in enumerate(ids):
    # Add to the corresponding dictionary
    lang_features[id_]['tfidf'] = pure_tfidf_matrix[idx]

In [60]:
print(pure_tfidf_matrix.shape)

(134, 1005)


#### Combine data rows to save features as CSV

In [61]:
# Copilot assisted code generation
rows = list()
for i, id in enumerate(ids):
    features = lang_features[id]
    row = {'id':id}

    # Add PHQ scores (outcome)
    row['PHQ_Score'] = phq_scores[i]

    # Get sentiments
    for name,score in features['sentiment'].items():
        row[name] = score
    
    # Get embeddings
    for i, val in enumerate(features['embedding']):
        row[f'embed_{i}'] = val
    
    # Get tfidf features
    for i, val in enumerate(features['tfidf']):
        row[f'tfidf_{i}'] = val
    
    rows.append(row)

# Save language features as csv
lang_df = pd.DataFrame(rows)
lang_df.to_csv('edaic_transcript_features.csv', index=False)

## Split data into 5 folds as instructed

## (b) Estimate depression severity using a decision tree